In [2]:
from pathlib import Path
import re
import unicodedata
from collections import Counter
from typing import Optional

In [3]:
# Configuration

INPUT_DIR = Path("../../data/articles")

ALLOWED_PUNCTUATION = {
    ".",
    ",",
    ":",
    ";",
    "?",
    "!",
    "-",
    "_",
    "(",
    ")",
    "[",
    "]",
    "{",
    "}",
    "'",
    '"',
    "/",
    "%",
}

In [4]:
# # Parse <s ...>...</s>
SENTENCE_PATTERN = re.compile(
    r'<s\s+'
    r'docid="(?P<docid>[^"]+)"\s+'
    r'num="(?P<num>[^"]+)"\s+'
    r'wdcount="(?P<wdcount>\d+)"\s+'
    r'type="(?P<type>[^"]+)"'
    r'>(?P<content>.*?)</s>',
    re.DOTALL,
)

def parse_sentence(line: str) -> Optional[dict]:
    """
    Parse một dòng dạng:

    <s docid="261" num="1" wdcount="12" type="title">Nội dung</s>

    Returns:
        dict nếu parse thành công.
        None nếu dòng không đúng format.
    """

    match = SENTENCE_PATTERN.search(line.strip())
    if not match:
        return None

    return {
        "docid": match.group("docid"),
        "num": int(match.group("num")),
        "wdcount": int(match.group("wdcount")),
        "type": match.group("type"),
        "content": match.group("content").strip(),
    }

In [5]:
# ============================================================
# Special character detection
# ============================================================

def get_special_characters(text: str) -> list[str]:
    """
    Tìm các ký tự đặc biệt trong text.

    Không coi là ký tự đặc biệt:
        - chữ cái Unicode, bao gồm tiếng Việt
        - chữ số
        - khoảng trắng
        - punctuation thông thường trong ALLOWED_PUNCTUATION

    Ví dụ có thể bị phát hiện: © ® ™ • → ★ emoji, ký tự control, ký tự Unicode lạ
    """

    special_chars = []
    for char in text:
        if char.isspace():
            continue
        if char.isalnum():
            continue
        if char in ALLOWED_PUNCTUATION:
            continue
        special_chars.append(char)

    return special_chars

def describe_character(char: str) -> str:
    """
    Lấy thông tin Unicode của ký tự.
    """

    code_point = f"U+{ord(char):04X}"

    name = unicodedata.name(char, "UNKNOWN")

    return f"{repr(char)} | {code_point} | {name}"

In [ ]:
# ============================================================
# Main analysis
# ============================================================
txt_files = sorted(INPUT_DIR.rglob("*.txt"))

if not txt_files:
    print(f"Không tìm thấy file txt trong: {INPUT_DIR.resolve()}")
    raise SystemExit(1)

print(f"Found {len(txt_files)} txt files")
print(f"Input directory: {INPUT_DIR.resolve()}")
print()

# ------------------------------------------------------------
# 1. Kiểm tra wdcount của dòng cuối mỗi file
# ------------------------------------------------------------
last_line_results = []
# ------------------------------------------------------------
# 2. Thống kê ký tự đặc biệt
# ------------------------------------------------------------
special_char_counter = Counter()
special_paragraphs = []

for file_path in txt_files:
    try:
        lines = file_path.read_text(encoding="utf-8").splitlines()
    except UnicodeDecodeError as error:
        print(f"[UTF-8 ERROR] {file_path}")
        print(error)
        continue
    # ========================================================
    # Tìm dòng cuối cùng hợp lệ
    # ========================================================
    last_sentence = None
    for line in reversed(lines):
        sentence = parse_sentence(line)
        if sentence is not None:
            last_sentence = sentence
            break

    if last_sentence is not None:
        last_line_results.append(
            {
                "path": file_path,
                "docid": last_sentence["docid"],
                "num": last_sentence["num"],
                "type": last_sentence["type"],
                "wdcount": last_sentence["wdcount"],
                "content": last_sentence["content"],
            }
        )

    # ========================================================
    # Scan toàn bộ sentence để tìm special chars
    # ========================================================

    for line in lines:
        sentence = parse_sentence(line)
        if sentence is None:
            continue

        content = sentence["content"]
        special_chars = get_special_characters(content)
        if not special_chars:
            continue

        special_char_counter.update(special_chars)
        special_paragraphs.append(
            {
                "path": file_path,
                "docid": sentence["docid"],
                "num": sentence["num"],
                "type": sentence["type"],
                "special_chars": sorted(set(special_chars)),
                "content": content,
            }
        )

# ============================================================
# Report: min / max wdcount
# ============================================================

print("=" * 80)
print("LAST SENTENCE WDCOUNT")
print("=" * 80)

if last_line_results:
    min_result = min(
        last_line_results,
        key=lambda item: item["wdcount"],
    )

    max_result = max(
        last_line_results,
        key=lambda item: item["wdcount"],
    )

    print("\nMIN wdcount")
    print("-" * 80)
    print(f"wdcount : {min_result['wdcount']}")
    print(f"path    : {min_result['path']}")
    print(f"docid   : {min_result['docid']}")
    print(f"num     : {min_result['num']}")
    print(f"type    : {min_result['type']}")
    print(f"content : {min_result['content']}")
    print("\nMAX wdcount")
    print("-" * 80)
    print(f"wdcount : {max_result['wdcount']}")
    print(f"path    : {max_result['path']}")
    print(f"docid   : {max_result['docid']}")
    print(f"num     : {max_result['num']}")
    print(f"type    : {max_result['type']}")
    print(f"content : {max_result['content']}")

# ============================================================
# Report: special characters
# ============================================================

print()
print("=" * 80)
print("SPECIAL CHARACTER STATISTICS")
print("=" * 80)

for char, count in special_char_counter.most_common():
    print(f"{describe_character(char):50} count={count}")

# ============================================================
# Report: paragraphs containing special chars
# ============================================================
print()
print("=" * 80)
print("SENTENCES CONTAINING SPECIAL CHARACTERS")
print("=" * 80)

for item in special_paragraphs:
    print()
    print(f"File    : {item['path']}")
    print(f"docid   : {item['docid']}")
    print(f"num     : {item['num']}")
    print(f"type    : {item['type']}")
    print("special :")
    for char in item["special_chars"]:
        print(f"    {describe_character(char)}")
    print(f"content : {item['content']}")

Found 812 txt files
Input directory: /Users/thangtran/Workplace/master_s_degree/information_retrieval/backend/data/articles

LAST SENTENCE WDCOUNT

MIN wdcount
--------------------------------------------------------------------------------
wdcount : 1
path    : ../../data/articles/1212.txt
docid   : 1212
num     : 7
type    : paragraph
content : Hi

MAX wdcount
--------------------------------------------------------------------------------
wdcount : 80
path    : ../../data/articles/645.txt
docid   : 645
num     : 11
type    : paragraph
content : C06 đánh giá việc tích hợp giấy chứng nhận quyền sử dụng đất trên VNeID còn nhiều vấn đề cần tiếp tục nghiên cứu, hoàn thiện. Dù quy định hiện hành đã xác định giấy chứng nhận điện tử có giá trị pháp lý như bản giấy khi đáp ứng đủ điều kiện, song còn thiếu hướng dẫn cụ thể với việc chuyển đổi từ bản giấy sang điện tử; nhất là với các hồ sơ cũ, thiếu thông tin.

SPECIAL CHARACTER STATISTICS
'&' | U+0026 | AMPERSAND                           co